#### 🤖 TopDev Auto Posting Tool
**Bấm ▶️ Run All để chạy toàn bộ quy trình tự động đăng tin tuyển dụng trên TopDev**
---

In [1]:
# ════════════════════════════════════════════════
# [1] CONFIG
# ════════════════════════════════════════════════
import os, sys, time, threading, asyncio
sys.path.insert(0, os.getcwd())

if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

JD_FILE    = 'data/JD.xlsx'
HEADLESS   = False
POST_LOGIN_WAIT = 420   # 7 phút (420s) chờ đăng nhập
JOB_WAIT   = 3

with open('.env', 'r', encoding='utf-8') as f:
    lines = [l.strip() for l in f.readlines() if l.strip()]
    EMAIL    = lines[0]
    PASSWORD = lines[1]

print('⚙️  CONFIG OK')

⚙️  CONFIG OK


In [2]:
# ════════════════════════════════════════════════
# [2] ĐỌC FILE EXCEL
# ════════════════════════════════════════════════
from topdev_helpers import read_excel, preview_jobs

jobs = read_excel(JD_FILE)
preview_jobs(jobs)

✅ Đọc được 1 tin tuyển dụng từ data/JD.xlsx

#######################################################
  DANH SÁCH JD (1 tin)
#######################################################
  [0] Chuyên viên Khách hàng Cá nhân - Khu vực Long Biên, Hà Nội (2026TD4510
       📍 Hà Nội  |  💰 9000000 – 35000000



In [3]:
# ════════════════════════════════════════════════
# [3] MỞ TRÌNH DUYỆT + ĐĂNG TIN (PRODUCTION)
# ════════════════════════════════════════════════
import sys, time, threading, asyncio
from playwright.sync_api import sync_playwright
from topdev_automation import post_single_job

results   = []
error_box = [None]

def run_playwright():
    if sys.platform == 'win32':
        asyncio.set_event_loop(asyncio.new_event_loop())
    try:
        with sync_playwright() as pw:
            context = pw.chromium.launch_persistent_context(
                user_data_dir='./browser_profile', headless=HEADLESS,
                args=['--start-maximized'], viewport={'width': 1440, 'height': 900}, locale='vi-VN'
            )
            page = context.pages[0] if context.pages else context.new_page()

            # ────────────────────────────────────────────
            # ĐI THẲNG VÀO DASHBOARD ĐỂ CHECK SESSION
            # ────────────────────────────────────────────
            DASH_URL = 'https://dash.topdev.vn/jobs'
            page.goto(DASH_URL, timeout=60000)
            try:
                page.wait_for_load_state('networkidle', timeout=15000)
            except:
                pass
            time.sleep(2)

            def _is_logged_in():
                """Kiểm tra page hiện tại đã vào dashboard chưa."""
                url = page.url.lower()
                # Debug: in URL hiện tại để dễ theo dõi
                print(f'  🔍 URL hiện tại: {page.url}')
                # Nếu đang ở Cloudflare "Just a moment..."
                try:
                    title = page.title().lower()
                    if any(kw in title for kw in ['just a moment', 'cloudflare', 'attention required']):
                        print(f'  ⏳ Đang chờ Cloudflare... (title: {page.title()})')
                        return False
                except:
                    pass
                # Nếu bị redirect về trang login → chưa đăng nhập
                if '/login' in url or '/auth' in url:
                    return False
                # Nếu URL chứa dash.topdev.vn hoặc employer → đã vào dashboard
                if 'dash.topdev.vn' in url or 'employer' in url or 'topdev.vn/app' in url:
                    return True
                # Nếu URL vẫn là dash.topdev.vn nhưng không có 'login' → OK
                if 'topdev.vn' in url and 'login' not in url:
                    return True
                return False

            if _is_logged_in():
                print('✅ Đã có dữ liệu đăng nhập từ trước. Vào thẳng form đăng tin, không phải chờ 7 phút!')
            else:
                print('🔑 Chưa đăng nhập hoặc đang ở phòng chờ (Cloudflare)...')
                # Nếu bị redirect về login, thử auto-fill credentials
                if '/login' in page.url.lower() or '/auth' in page.url.lower():
                    try:
                        page.wait_for_selector("input[placeholder='Account']", state='visible', timeout=10000)
                        time.sleep(1)
                        page.get_by_placeholder('Account').fill(EMAIL)
                        page.get_by_placeholder('Password').fill(PASSWORD)
                        for s in ["button[type='submit']", "button:has-text('Log in')"]:
                            try: page.locator(s).first.click(timeout=2000); break
                            except: continue
                    except:
                        print('  ⚠️ Vui lòng hoàn tất xác thực hoặc đăng nhập thủ công!')

                print('⏳ Hệ thống sẽ chờ tối đa 7 phút để bạn vượt qua xác thực/đăng nhập...')
                for i in range(420, 0, -1):
                    if _is_logged_in():
                        print('\n✅ Đã phát hiện đăng nhập thành công, bỏ qua thời gian chờ còn lại!')
                        break

                    mins, secs = divmod(i, 60)
                    sys.stdout.write(f'\rCòn lại: {mins:02d}:{secs:02d}  ')
                    sys.stdout.flush()
                    time.sleep(1)

                if not _is_logged_in():
                    print('\n❌ Đã hết 7 phút mà vẫn chưa đăng nhập thành công. Dừng lại.')
                    context.close()
                    return

            # ────────────────────────────────────────────
            # BẮT ĐẦU ĐĂNG TIN
            # ────────────────────────────────────────────
            total = len(jobs)
            print('\n🚀 Bắt đầu chạy %d tin\n' % total)

            for idx, job in enumerate(jobs):
                print('\n[%d/%d] %s' % (idx+1, total, '─'*40))
                ok = post_single_job(page, job)
                results.append({'title': job.get('Tên hiển thị trên website MB', ''), 'ok': ok})
                if idx < total - 1:
                    time.sleep(JOB_WAIT)

            ok_count = sum(1 for r in results if r['ok'])
            print('\n📊 KẾT QUẢ: %d/%d thành công' % (ok_count, total))
            for r in results:
                print('  %s %s' % ('✅' if r['ok'] else '❌', r['title'][:70]))

            # Lưu cookies trước khi đóng (đảm bảo session persist)
            try:
                context.storage_state(path='./browser_profile/storage_state.json')
                print('💾 Đã lưu session để lần sau không cần đăng nhập lại.')
            except:
                pass
            context.close()
            print('✅ Hoàn tất!')
    except Exception as e:
        error_box[0] = e
        print('💥 %s' % e)

print('⏳ Đang khởi động...\n')
t = threading.Thread(target=run_playwright, daemon=True)
t.start()
t.join()
if error_box[0]: raise error_box[0]

⏳ Đang khởi động...

  🔍 URL hiện tại: https://dash.topdev.vn/jobs
✅ Đã có dữ liệu đăng nhập từ trước. Vào thẳng form đăng tin, không phải chờ 7 phút!

🚀 Bắt đầu chạy 1 tin


[1/1] ────────────────────────────────────────

═══════════════════════════════════════════════════════
🚀 Bắt đầu đăng tin: Chuyên viên Khách hàng Cá nhân - Khu vực Long Biên, Hà Nội (2026TD451031)
═══════════════════════════════════════════════════════

───────────────────────────────────────────────────────
📌 Bước 1: Điều hướng trang tạo job
  ✅ Đã vào trang tạo job

───────────────────────────────────────────────────────
📌 Bước 2a: Chọn Package
  ✅ Package: Top Job Unlimited - MBBank

───────────────────────────────────────────────────────
📌 Bước 2b: Thông tin cơ bản
   Chuyên viên Khách hàng Cá nhân - Khu vực Long Biên, Hà Nội (2026TD451031)
  ✅ Title: Chuyên viên Khách hàng Cá nhân - Khu vực Long Biên, Hà Nội (
  ⏳ Job Category: Business, Finance
  ✅ Job Category: Business, Finance
  ⏳ Role: Sales / Business 